In [ ]:
# ============================================================
# CELL 1 — Load Data
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

results      = pd.read_csv('../../data/results.csv')
goalscorers  = pd.read_csv('../../data/goalscorers.csv')
shootouts    = pd.read_csv('../../data/shootouts.csv')
former_names = pd.read_csv('../../data/former_names.csv')

results['date']     = pd.to_datetime(results['date'])
goalscorers['date'] = pd.to_datetime(goalscorers['date'])
shootouts['date']   = pd.to_datetime(shootouts['date'])

print('Loaded!')
print('results     :', results.shape)
print('goalscorers :', goalscorers.shape)
print('shootouts   :', shootouts.shape)
print('former_names:', former_names.shape)


In [ ]:
# ============================================================
# CELL 2 — Fix Country Names
# Some countries changed names (e.g. West Germany -> Germany)
# We must unify them before doing any calculation
# ============================================================
name_map = dict(zip(former_names['former'], former_names['current']))

def normalize(name):
    return name_map.get(name, name)

results['home_team']     = results['home_team'].apply(normalize)
results['away_team']     = results['away_team'].apply(normalize)

goalscorers['home_team'] = goalscorers['home_team'].apply(normalize)
goalscorers['away_team'] = goalscorers['away_team'].apply(normalize)
goalscorers['team']      = goalscorers['team'].apply(normalize)

shootouts['home_team']   = shootouts['home_team'].apply(normalize)
shootouts['away_team']   = shootouts['away_team'].apply(normalize)
shootouts['winner']      = shootouts['winner'].apply(normalize)

print('Country names normalized!')

In [ ]:
# ============================================================
# CELL 3 — Drop rows with missing scores, create target label
# Label = who won the match
#   0 = home team won
#   1 = draw (tie)
#   2 = away team won
# ============================================================
results = results.dropna(subset=['home_score', 'away_score']).copy()

def get_outcome(row):
    if row['home_score'] > row['away_score']:
        return 0  # home win
    elif row['home_score'] == row['away_score']:
        return 1  # draw
    else:
        return 2  # away win

results['label'] = results.apply(get_outcome, axis=1)

print(results['label'].value_counts())
print('0 = home win, 1 = draw, 2 = away win')

In [ ]:
# ============================================================
# CELL 4 — Build team stats lookup table
# For each team, compute (using ALL past matches up to a date):
#   - win rate (how often they win)
#   - avg goals scored
#   - avg goals conceded
#   - shootout win rate
# We only use matches BEFORE the current match (no data leakage)
# ============================================================
results_sorted = results.sort_values('date').reset_index(drop=True)

def team_stats(team, before_date, n=20):
    past = results_sorted[
        (results_sorted['date'] < before_date) &
        ((results_sorted['home_team'] == team) | (results_sorted['away_team'] == team))
    ].tail(n)

    if len(past) == 0:
        return {'win_rate': 0.5, 'avg_scored': 1.0, 'avg_conceded': 1.0, 'n_matches': 0}

    wins, scored, conceded = [], [], []
    for _, m in past.iterrows():
        if m['home_team'] == team:
            wins.append(1 if m['home_score'] > m['away_score'] else 0)
            scored.append(m['home_score'])
            conceded.append(m['away_score'])
        else:
            wins.append(1 if m['away_score'] > m['home_score'] else 0)
            scored.append(m['away_score'])
            conceded.append(m['home_score'])

    return {
        'win_rate':     np.mean(wins),
        'avg_scored':   np.mean(scored),
        'avg_conceded': np.mean(conceded),
        'n_matches':    len(past)
    }

def shootout_win_rate(team, before_date):
    past = shootouts[
        (shootouts['date'] < before_date) &
        ((shootouts['home_team'] == team) | (shootouts['away_team'] == team))
    ]
    if len(past) == 0:
        return 0.5
    return (past['winner'] == team).sum() / len(past)

print('Functions defined!')

In [ ]:
# ============================================================
# CELL 6 — Split into Train and Test sets
# Train on older matches, test on recent matches
# This is more realistic than random split for time-series data
# ============================================================
from sklearn.model_selection import train_test_split

FEATURES = [
    'is_neutral', 'year', 'month',
    'home_win_rate', 'home_avg_scored', 'home_avg_conceded', 'home_n_matches', 'home_shootout_wr',
    'away_win_rate', 'away_avg_scored', 'away_avg_conceded', 'away_n_matches', 'away_shootout_wr',
]

X = feat[FEATURES]
y = feat['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Train size:', X_train.shape)
print('Test size :', X_test.shape)

In [ ]:
# ============================================================
# CELL 7 — Train the Model
# Random Forest: builds many decision trees and votes
# ============================================================
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

print('Model trained!')

In [ ]:
# ============================================================
# CELL 8 — Evaluate the Model
# ============================================================
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

y_pred = model.predict(X_test)

print('Accuracy:', round(accuracy_score(y_test, y_pred) * 100, 2), '%')
print()
print(classification_report(y_test, y_pred, target_names=['Home Win', 'Draw', 'Away Win']))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=['Home Win', 'Draw', 'Away Win'],
            yticklabels=['Home Win', 'Draw', 'Away Win'],
            cmap='Blues')
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

# Feature importance
imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
imp.plot(kind='barh', figsize=(7, 5), title='Feature Importance')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 9 — Predict a Match Between Two Countries
# ============================================================
def predict_match(home_team, away_team, is_neutral=0):
    today = pd.Timestamp.today()

    h = team_stats(home_team, today)
    a = team_stats(away_team, today)

    input_row = pd.DataFrame([{
        'is_neutral':        is_neutral,
        'year':              today.year,
        'month':             today.month,
        'home_win_rate':     h['win_rate'],
        'home_avg_scored':   h['avg_scored'],
        'home_avg_conceded': h['avg_conceded'],
        'home_n_matches':    h['n_matches'],
        'home_shootout_wr':  shootout_win_rate(home_team, today),
        'away_win_rate':     a['win_rate'],
        'away_avg_scored':   a['avg_scored'],
        'away_avg_conceded': a['avg_conceded'],
        'away_n_matches':    a['n_matches'],
        'away_shootout_wr':  shootout_win_rate(away_team, today),
    }])

    pred  = model.predict(input_row)[0]
    proba = model.predict_proba(input_row)[0]
    label = {0: 'Home Win', 1: 'Draw', 2: 'Away Win'}

    print(f'\n  {home_team}  vs  {away_team}')
    print(f'  Venue     : {"Neutral" if is_neutral else home_team + " home ground"}')
    print(f'  Prediction: {label[pred]}')
    print(f'  {home_team} wins : {proba[0]*100:.1f}%')
    print(f'  Draw        : {proba[1]*100:.1f}%')
    print(f'  {away_team} wins : {proba[2]*100:.1f}%')

# ---- Try it here ----
predict_match('Brazil', 'Argentina')
predict_match('France', 'England')
predict_match('Spain', 'Germany', is_neutral=1)

In [ ]:
predict_match('Argentina', 'Iran')

In [ ]:
predict_match('New Zealand', 'Egypt')

---
## Task 1 — Win / Draw / Loss Prediction
Compare multiple classifiers and keep the best one.

In [ ]:
# ============================================================
# CELL 10 — Task 1: Compare Multiple Classifiers
# ============================================================
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import time

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('xgboost not installed — skipping. Run: pip install xgboost')

classifiers = {
    'Random Forest':  RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boost': GradientBoostingClassifier(n_estimators=200, random_state=42),
    'Logistic Reg':   LogisticRegression(max_iter=1000, random_state=42),
}
if HAS_XGB:
    classifiers['XGBoost'] = XGBClassifier(n_estimators=200, random_state=42, eval_metric='mlogloss', verbosity=0)

clf_results = {}
for name, clf in classifiers.items():
    start = time.time()
    clf.fit(X_train, y_train)
    elapsed = time.time() - start
    acc = accuracy_score(y_test, clf.predict(X_test))
    clf_results[name] = {'model': clf, 'accuracy': acc}
    print(f'{name:20s}: {acc*100:.2f}%  ({elapsed:.1f}s)')

best_clf_name = max(clf_results, key=lambda k: clf_results[k]['accuracy'])
best_clf      = clf_results[best_clf_name]['model']
print(f'\nBest for outcome prediction: {best_clf_name}  ({clf_results[best_clf_name]["accuracy"]*100:.2f}%)')

---
## Task 2 — Exact Score Prediction
Train regression models to predict `home_score` and `away_score` separately, then round to nearest integer.

In [ ]:
# ============================================================
# CELL 11 — Prepare Regression Targets
# Merge actual scores into the feature table.
# Use the same train/test split (random_state=42).
# ============================================================
score_cols = results_sorted[['date', 'home_team', 'away_team', 'home_score', 'away_score']]
feat_reg   = feat.merge(score_cols, on=['date', 'home_team', 'away_team'], how='left')

X_reg  = feat_reg[FEATURES]
y_home = feat_reg['home_score']
y_away = feat_reg['away_score']

idx_train, idx_test = train_test_split(range(len(X_reg)), test_size=0.2, random_state=42)

X_train_r    = X_reg.iloc[idx_train].reset_index(drop=True)
X_test_r     = X_reg.iloc[idx_test].reset_index(drop=True)
y_home_train = y_home.iloc[idx_train].reset_index(drop=True)
y_home_test  = y_home.iloc[idx_test].reset_index(drop=True)
y_away_train = y_away.iloc[idx_train].reset_index(drop=True)
y_away_test  = y_away.iloc[idx_test].reset_index(drop=True)

print('Regression targets ready!')
print(f'Train: {X_train_r.shape}   Test: {X_test_r.shape}')

In [ ]:
# ============================================================
# CELL 12 — Task 2: Compare Multiple Regressors
# Train one model for home_score and one for away_score each.
# Best = highest exact score accuracy (both goals correct).
# ============================================================
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error

try:
    from xgboost import XGBRegressor
    HAS_XGB_R = True
except ImportError:
    HAS_XGB_R = False
    print('xgboost not installed — skipping. Run: pip install xgboost')

regressors = {
    'Random Forest':  RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boost': GradientBoostingRegressor(n_estimators=200, random_state=42),
    'Ridge':          Ridge(),
}
if HAS_XGB_R:
    regressors['XGBoost'] = XGBRegressor(n_estimators=200, random_state=42, verbosity=0)

reg_results = {}
for name, base in regressors.items():
    reg_home = clone(base)
    reg_away = clone(base)

    reg_home.fit(X_train_r, y_home_train)
    reg_away.fit(X_train_r, y_away_train)

    home_pred = reg_home.predict(X_test_r)
    away_pred = reg_away.predict(X_test_r)

    mae_home  = mean_absolute_error(y_home_test, home_pred)
    mae_away  = mean_absolute_error(y_away_test, away_pred)
    exact_acc = np.mean(
        (np.round(home_pred) == y_home_test.values) &
        (np.round(away_pred) == y_away_test.values)
    )

    reg_results[name] = {
        'home_model': reg_home,
        'away_model': reg_away,
        'mae_home':   mae_home,
        'mae_away':   mae_away,
        'exact_acc':  exact_acc,
    }
    print(f'{name:20s}  MAE home={mae_home:.2f}  away={mae_away:.2f}  exact score={exact_acc*100:.1f}%')

best_reg_name = max(reg_results, key=lambda k: reg_results[k]['exact_acc'])
best_reg      = reg_results[best_reg_name]
print(f'\nBest for score prediction: {best_reg_name}  (exact score acc = {best_reg["exact_acc"]*100:.1f}%)')

In [ ]:
# ============================================================
# CELL 13 — Save All Best Models
# ============================================================
import joblib, os

os.makedirs('../../dataII/V1', exist_ok=True)

feat.to_csv('../../dataII/V1/features.csv', index=False)
joblib.dump(best_clf,               '../../dataII/V1/model_outcome.pkl')
joblib.dump(best_reg['home_model'], '../../dataII/V1/model_home_score.pkl')
joblib.dump(best_reg['away_model'], '../../dataII/V1/model_away_score.pkl')

print('Saved:')
print('  features.csv         — feature table')
print(f'  model_outcome.pkl    — {best_clf_name}')
print(f'  model_home_score.pkl — {best_reg_name} (home goals)')
print(f'  model_away_score.pkl — {best_reg_name} (away goals)')


In [ ]:
# ============================================================
# CELL 14 — Full Prediction: Outcome + Exact Score
# ============================================================
def predict_match_full(home_team, away_team, is_neutral=0):
    today = pd.Timestamp.today()

    h = team_stats(home_team, today)
    a = team_stats(away_team, today)

    row = pd.DataFrame([{
        'is_neutral':        is_neutral,
        'year':              today.year,
        'month':             today.month,
        'home_win_rate':     h['win_rate'],
        'home_avg_scored':   h['avg_scored'],
        'home_avg_conceded': h['avg_conceded'],
        'home_n_matches':    h['n_matches'],
        'home_shootout_wr':  shootout_win_rate(home_team, today),
        'away_win_rate':     a['win_rate'],
        'away_avg_scored':   a['avg_scored'],
        'away_avg_conceded': a['avg_conceded'],
        'away_n_matches':    a['n_matches'],
        'away_shootout_wr':  shootout_win_rate(away_team, today),
    }])

    # Task 1: outcome
    outcome = best_clf.predict(row)[0]
    proba   = best_clf.predict_proba(row)[0]
    label   = {0: 'Home Win', 1: 'Draw', 2: 'Away Win'}

    # Task 2: score
    home_g = max(0, int(round(best_reg['home_model'].predict(row)[0])))
    away_g = max(0, int(round(best_reg['away_model'].predict(row)[0])))

    print(f'\n  {home_team}  vs  {away_team}')
    print(f'  Venue          : {"Neutral" if is_neutral else home_team + " home ground"}')
    print(f'  Predicted Score: {home_team} {home_g} - {away_g} {away_team}')
    print(f'  Outcome        : {label[outcome]}')
    print(f'  {home_team} win  : {proba[0]*100:.1f}%')
    print(f'  Draw           : {proba[1]*100:.1f}%')
    print(f'  {away_team} win  : {proba[2]*100:.1f}%')

# ---- Try it ----
predict_match_full('Brazil', 'Argentina')
predict_match_full('France', 'England')
predict_match_full('Spain', 'Germany', is_neutral=1)

In [ ]:
# predict_match_full('United States', 'Paraguay')
predict_match_full('Mexico', 'South Africa')
predict_match_full('South Korea', 'Czech Republic')
predict_match_full('Canada', 'Bosnia-Herzegovina')

In [ ]:
predict_match_full('Argentina', 'Austria')
predict_match_full('France', 'Iraq')


In [ ]:
predict_match_full('Norway', 'Senegal')
predict_match_full('Jordan', 'Algeria')

In [ ]:
predict_match_full('Portugal', 'Uzbekistan')
predict_match_full('England', 'Ghana')